In [2]:
import re
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

***ОТДЕЛЬНЫЙ БЛОКНОТ ПОД ВЫДЕЛЕНИЯ КОЛОНОК-ИНГРЕДИЕНТОВ И ОЧИСТКИ ДФ ОТ ДУБЛИКАТОВ СТРОК ДЛЯ ПОСЛЕДУЮЩЕЙ ИНТЕГРАЦИИ В ПАЙПЛАЙН***

In [3]:
with open(
    "../materials/Epicurious_dataset/full_format_recipes.json", "r", encoding="utf-8"
) as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data)
print(f"Исходный размер файла с рецептами: {df_raw.shape[0]} строк.")

Исходный размер файла с рецептами: 20130 строк.


In [4]:
df_raw.head(1)

,directions,fat,date,categories,calories,desc,protein,rating,title,ingredients,sodium
0,"[1. Place the stock, lentils, celery, carrot, ...",7.0,2006-09-01T04:00:00.000Z,"[Sandwich, Bean, Fruit, Tomato, turkey, Vegeta...",426.0,NaN,30.0,2.5,"Lentil, Apple, and Turkey Wrap","[4 cups low-sodium vegetable or chicken stock,...",559.0


In [5]:
df_clean = df_raw.dropna(subset=["ingredients"]).copy().reset_index(drop=True)
print(f"размер очищенного от na в ингредиентах датафрейма: {len(df_clean)} строк.")

размер очищенного от na в ингредиентах датафрейма: 20111 строк.


In [6]:
ingr = df_clean.loc[0, "ingredients"]
ingr

['4 cups low-sodium vegetable or chicken stock',
 '1 cup dried brown lentils',
 '1/2 cup dried French green lentils',
 '2 stalks celery, chopped',
 '1 large carrot, peeled and chopped',
 '1 sprig fresh thyme',
 '1 teaspoon kosher salt',
 '1 medium tomato, cored, seeded, and diced',
 '1 small Fuji apple, cored and diced',
 '1 tablespoon freshly squeezed lemon juice',
 '2 teaspoons extra-virgin olive oil',
 'Freshly ground black pepper to taste',
 '3 sheets whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas',
 '3/4 pound turkey breast, thinly sliced',
 '1/2 head Bibb lettuce']

In [7]:
# обработка самописным парсером с помощью регулярок и циклов отбрасывания ненужных слов stop_words типо 'cup', 'cups', 'stalk', 'stalks', 'sprig', 'sprigs', 'teaspoon', 'teaspoons', 'tsp', 'tsps', 'tablespoon', 'tablespoons', 'tbsp', 'tbsps', 'pound', 'pounds', 'lb', 'lbs', 'head', 'heads', 'sheet', 'sheets', 'large', 'medium', 'small', 'inch', 'oz', 'ounces', 'clove', 'cloves', 'of', 'low', 'sodium', 'dried', 'fresh', 'freshly', 'squeezed', 'ground', 'to', 'taste', 'chopped', 'sliced', 'diced', 'peeled', 'cored', 'seeded', 'thinly', 'cut', 'in', 'half', 'crosswise', 'and', 'with', 'for' и цифр тип 1, 1/3..
# не привели к достаточному качеству обработки строк

# Вместо того чтобы постоянно раздувать список stop_words, был применен метод Lemmatization (Лемматизации) / POS-Tagging (Part of Speech) с помощью самой популярной библиотеки для анализа текста — spaCy. Она работает на базе обученной нейросети и умеет 1) приводить любое слово к его начальной форме (лемме): tomatoes -> tomato, cloves -> clove, 2) определять часть речи, за счет чего заберем из строки только существительные, а все граммовки, цифры, бренды и прилагательные отбросим.

# библиотека и легкая языковая модель
# !pip install spacy
# !python -m spacy download en_core_web_sm

In [8]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])


def clean_recipe_ingredients_list_nlp_v6(ingredients_list):
    if not isinstance(ingredients_list, list):
        return []

    cleaned_recipe_output = []

    garbage_measures = {
        "cup",
        "cups",
        "teaspoon",
        "teaspoons",
        "tablespoon",
        "tablespoons",
        "tsp",
        "tbsp",
        "pound",
        "pounds",
        "clove",
        "cloves",
        "inch",
        "ounce",
        "ounces",
        "oz",
        "piece",
        "pieces",
        "can",
        "cans",
        "package",
        "packages",
        "stalk",
        "stalks",
        "sprig",
        "sprigs",
        "head",
        "heads",
        "sheet",
        "sheets",
        "bulb",
        "bulbs",
        "pinch",
        "pinches",
        "leaf",
        "leaves",
        "fillet",
        "fillets",
        "low",
        "medium",
        "small",
        "large",
        "half",
        "crosswise",
        "purpose",
        "style",
        "sodium",
        "ground",
        "taste",
        "dice",
        "slice",
        "slices",
        "chunk",
        "chunks",
        "baking",
        "trimmed",
        "boneless",
        "center",
        "loin",
        "sinew",
        "removed",
        "chilled",
        "well",
        "all",
        "discarded",
        "feathery",
        "garnish",
        "reserved",
        "packed",
        "finely",
        "sometimes",
        "called",
        "anise",
        "grain",
        "strip",
        "store",
    }

    for raw_item in ingredients_list:
        if not isinstance(raw_item, str):
            continue

        # убираем неразрывные пробелы, превращая их в обычные пробелы
        item_text = raw_item.replace("\xa0", " ")

        # разделяем слипшиеся цифры и буквы (например, 1/2cup -> 1/2 cup)
        item_text = re.sub(r"(\d+)([a-zA-Z])", r"\1 \2", item_text)

        # убираем все конструкции в круглых скобках
        item_text = re.sub(r"\(.*?\)", "", item_text)

        # разделяем альтернативные ингредиенты по союзу or
        raw_parts = re.split(r"[\s,]*\bor\b[\s,]*", item_text, flags=re.IGNORECASE)

        for part in raw_parts:
            # разрезаем строку по знакам препинания и глаголам кулинарной обработки
            sub_parts = re.split(
                r"[,;]|\b(cut|chopped|peeled|discarded|minced|crushed|halved)\b",
                part,
                flags=re.IGNORECASE,
            )

            # исправлено: безопасно фильтруем пустые элементы и склеиваем строку обратно через пробел
            clean_words_list = [p.strip() for p in sub_parts if p]
            part_clean = " ".join(clean_words_list).replace("-", " ").lower().strip()

            if not part_clean:
                continue

            doc = nlp(part_clean)

            # извлекаем лемматизированные существительные
            good_words = [
                token.lemma_
                for token in doc
                if token.pos_ in ["NOUN", "PROPN"] and len(token.lemma_) > 2
            ]
            filtered_words = [w for w in good_words if w not in garbage_measures]

            # дополнительно вырезаем сиротливые цифры и дроби, которые мог пропустить spacy
            filtered_words = [
                w for w in filtered_words if not re.match(r"^[\d./½¼¾%–]+$", w)
            ]

            if filtered_words:
                cleaned_str = " ".join(filtered_words).strip()
                if (
                    cleaned_str
                    and cleaned_str not in garbage_measures
                    and cleaned_str not in cleaned_recipe_output
                ):
                    cleaned_recipe_output.append(cleaned_str)

    return cleaned_recipe_output

In [9]:
tqdm.pandas(desc="nlp анализ")

print("запуск глобального анализа кулинарного словаря...")

# применение очистки v6 с отображением честного счетчика строк
df_clean["ingredients_clean"] = df_clean["ingredients"].progress_apply(
    clean_recipe_ingredients_list_nlp_v6
)

# сбор уникальных слов в множество
global_unique_ingredients = set()

# запуск легкого прогресс бара для сбора перечня ингредиентов
for recipe_list in tqdm(
    df_clean["ingredients_clean"], desc="сбор перечня ингредиентов"
):
    if isinstance(recipe_list, list):
        for ingredient in recipe_list:
            global_unique_ingredients.add(ingredient)

# сортировка итогового алфавитного словаря
final_unique_list = sorted(list(global_unique_ingredients))

# вывод результатов на экран
print("\nитоги глобального анализа")
print(f"всего уникальных продуктов извлечено из json: {len(final_unique_list)}")
print("-" * 50)
print("первые продукты в алфавитном порядке для контроля качества:")
print(final_unique_list[:30])
print("\nпоследние продукты в алфавитном порядке:")
print(final_unique_list[-30:])

запуск глобального анализа кулинарного словаря...


nlp анализ:   0%|          | 0/20111 [00:00<?, ?it/s]

сбор перечня ингредиентов:   0%|          | 0/20111 [00:00<?, ?it/s]


итоги глобального анализа
всего уникальных продуктов извлечено из json: 16119
--------------------------------------------------
первые продукты в алфавитном порядке для контроля качества:
['1⁄4 pepper', 'absinthe', 'acacia honey', 'accompaniment', 'accompaniment almond cake', 'accompaniment almond cooky fraîche cream', 'accompaniment almond macaroon', 'accompaniment almond meringue', 'accompaniment almond milk drizzling', 'accompaniment amaretto sauce', 'accompaniment apple celery salad', 'accompaniment applesauce', 'accompaniment apricot caramel sauce', 'accompaniment avocado', 'accompaniment avocado lime juice romaine onion lime wedge oregano', 'accompaniment bacon', 'accompaniment bacon toast', 'accompaniment bag baby green herb salad mix', 'accompaniment bagel chip', 'accompaniment baguette', 'accompaniment baguette dijon mustard', 'accompaniment baguette lemon wedge', 'accompaniment baguette marrow salt dijon mustard', 'accompaniment baguette oil', 'accompaniment baguette oven o

In [ ]:
df_orig = pd.read_csv("../materials/Epicurious_dataset/epi_r.csv")

# отбрасываем служебные не кулинарные колонки
first_drops = ["title", "rating", "calories", "protein", "fat", "sodium"]

In [11]:
print("nlp фильтрация колонок")

# берем все потенциальные признаки из оригинального epi_r.csv (без first_drops)
all_potential_ingredients = [col for col in df_orig.columns if col not in first_drops]

valid_ingredient_columns = []

for col in all_potential_ingredients:
    col_clean = col.lower().strip()

    # проверка имени колонки целиком (спасает 'maple syrup')
    if col_clean in final_unique_list:
        valid_ingredient_columns.append(col)
        continue

    # проверка сложных перечислений (milk/cream, jam or jelly)
    # разбиваем имя колонки на отдельные слова, убирая слэши и дефисы
    words_in_col = col_clean.replace("/", " ").replace("-", " ").split()

    # исключаем служебные союзы
    meaningful_words = [w for w in words_in_col if w not in ["or", "and"]]

    if not meaningful_words:
        continue

    # жесткий фильтр мусора: если в названии есть цифры (22) или технический мусор - пропускаем
    has_digits = any(w.isdigit() for w in meaningful_words)
    has_garbage_words = any(
        w in ["minute", "meals", "recipes", "groceries", "days"]
        for w in meaningful_words
    )

    if has_digits or has_garbage_words:
        continue

    # если все оставшиеся слова подтверждены списком из json - сохраняем
    # для 'jam or jelly' проверит 'jam' и 'jelly'. оба есть в json по отдельности -> True
    if all(word in final_unique_list for word in meaningful_words):
        valid_ingredient_columns.append(col)

print(f"количество колонок, подтвержденных nlp: {len(valid_ingredient_columns)}")

nlp фильтрация колонок
количество колонок, подтвержденных nlp: 366


In [12]:
print(f"количество колонок, подтвержденных nlp: {len(valid_ingredient_columns)}")
print(f"исходное число колонок в оригинальном epi_r.csv: {df_orig.shape}")
print(
    f"количество чистых ингредиентов, прошедших nlp валидацию: {len(valid_ingredient_columns)}"
)
print("-" * 50)
print(
    f"первые 30 сохраненных колонок-продуктов для контроля: {sorted(valid_ingredient_columns)[:30]}"
)
print(
    f"последние 30 сохраненных колонок-продуктов для контроля: {sorted(valid_ingredient_columns)[-30:]}"
)

количество колонок, подтвержденных nlp: 366
исходное число колонок в оригинальном epi_r.csv: (20052, 680)
количество чистых ингредиентов, прошедших nlp валидацию: 366
--------------------------------------------------
первые 30 сохраненных колонок-продуктов для контроля: ['almond', 'amaretto', 'anchovy', 'apple', 'apple juice', 'apricot', 'artichoke', 'arugula', 'asparagus', 'avocado', 'bacon', 'banana', 'barley', 'basil', 'bass', 'bean', 'beef', 'beef rib', 'beef shank', 'beef tenderloin', 'beer', 'beet', 'bell pepper', 'berry', 'biscuit', 'blackberry', 'blender', 'blueberry', 'boil', 'bok choy']
последние 30 сохраненных колонок-продуктов для контроля: ['tomato', 'tortillas', 'trout', 'tuna', 'turkey', 'turnip', 'vanilla', 'veal', 'vegan', 'vegetable', 'vermont', 'vermouth', 'vinegar', 'vodka', 'waffle', 'walnut', 'wasabi', 'watercress', 'watermelon', 'whiskey', 'white wine', 'whole wheat', 'wild rice', 'wine', 'winter', 'wisconsin', 'yellow squash', 'yogurt', 'yuca', 'zucchini']


In [13]:
words = [
    "zucchini",
    "ziti",
    "zinfandel",
    "onion",
    "lentil",
    "jam or jelly",
    "milk/cream",
]
print("проверка наличия чистых ингредиентов в словаре:")
for word in words:
    is_present = word in valid_ingredient_columns
    print(f"- {word}: {'присутствует' if is_present else 'отсутствует'}")


проверка наличия чистых ингредиентов в словаре:
- zucchini: присутствует
- ziti: отсутствует
- zinfandel: отсутствует
- onion: присутствует
- lentil: присутствует
- jam or jelly: присутствует
- milk/cream: присутствует


In [14]:
df_orig = pd.read_csv("../materials/Epicurious_dataset/epi_r.csv")

# проверяем как эти слова записаны в оригинальном файле
print([c for c in df_orig.columns if "ziti" in c.lower()])
print([c for c in df_orig.columns if "zinfandel" in c.lower()])

# то есть все норм- в оригинальном epi_r.csv нет вообще 'ziti', 'zinfandel', они только в json рецептах

[]
[]


все еще остались штаты, так как в df_orig (epi_r.csv) есть колонки ['maple syrup', 'vermont'],  которые в ингредиентах изначально существуют как vermont maple syrup (привет "pour 1 cup of Vermont maple syrup into the pan" из рецептов json) и также ['cheddar', 'wisconsin'] которые изначльно Wisconsin cheddar (привет "top with 2 cups of shredded Wisconsin cheddar" из рецептов json)

In [15]:
v = [
    col
    for col in valid_ingredient_columns
    if "syrup" in col.lower()
    or "maple syrup" in col.lower()
    or "vermont" in col.lower()
]

print(f" в обработанном датафреме остались колонки {v}")

v2 = [
    col
    for col in df_orig.columns
    if "syrup" in col.lower()
    or "maple syrup" in col.lower()
    or "vermont" in col.lower()
]
print(f" в оригинальном датафреме epi_r.csv есть колонки {v2}")

 в обработанном датафреме остались колонки ['maple syrup', 'vermont']
 в оригинальном датафреме epi_r.csv есть колонки ['maple syrup', 'vermont']


In [16]:
w = [
    col
    for col in valid_ingredient_columns
    if "wisconsin cheddar" in col.lower()
    or "cheddar" in col.lower()
    or "wisconsin" in col.lower()
]

print(f" в обработанном датафреме остались колонки {w}")

w2 = [
    col
    for col in df_orig.columns
    if "wisconsin cheddar" in col.lower()
    or "cheddar" in col.lower()
    or "wisconsin" in col.lower()
]
print(f" в оригинальном датафреме epi_r.csv есть колонки {w2}")

 в обработанном датафреме остались колонки ['cheddar', 'wisconsin']
 в оригинальном датафреме epi_r.csv есть колонки ['cheddar', 'wisconsin']


In [17]:
import pycountry

print("запуск финальной доочистки списка признаков от штатов и сезонов...")

# теперь мы можем писать и короткие слова-маркеры (thermometer), и дефисные (pan-fry)
perfect_stop_words = {
    "winter",
    "fall",
    "summer",
    "spring",
    "season",
    "seasons",
    "year",
    "night",
    "eve",
    "reunion",
    "new year",
    "thanksgiving",
    "christmas",
    "fourth of july",
    "easter",
    "wedding",
    "anniversary",
    "dinner",
    "lunch",
    "breakfast",
    "appetizer",
    "beverage",
    "side",
    "snack",
    "leftovers",
    "microwave",
    "slow cooker",
    "mortar and pestle",
    "cookbooks",
    "pan-fry",
    "pan fry",
    "fruit",
    "cake",
    "thermometer",
    "day",
    "fry",
    "casserole",
    "gratin",
    "wok",
}

# автоматический сбор названий всех географических штатов сша из pycountry в нижнем регистре
us_states = set(
    sub.name.lower().strip() for sub in pycountry.subdivisions.get(country_code="US")
)

# объединение географического и календарного шума в один черный список
final_blacklist = perfect_stop_words.union(us_states)

perfect_ingredient_columns = []

for col in valid_ingredient_columns:
    col_clean = col.lower().strip()

    # первая линия защиты: строгая проверка строки целиком
    if col_clean in final_blacklist:
        continue

    # вторая линия защиты: разбиваем строку СТРОГО по пробелам и слэшам, не трогая дефисы
    # 'candy thermometer' -> ['candy', 'thermometer']
    # 'pan-fry' -> ['pan-fry'] (останется целым и защитит дефисную логику)
    words_in_col = col_clean.replace("/", " ").split()

    # проверяем, совпало ли хоть одно отдельное слово со стоп-листом (поймает одиночный 'thermometer')
    is_blacklist_word = any(word in final_blacklist for word in words_in_col)

    if not is_blacklist_word:
        perfect_ingredient_columns.append(col)

print("\n=== ИТОГОВЫЙ ЭТАЛОННЫЙ КУЛИНАРНЫЙ СПИСОК ===")
print(f"было... колок после точечной nlp валидации: {len(valid_ingredient_columns)}")
print(f"осталось ингредиентов-столбцов для модели: {len(perfect_ingredient_columns)}")

запуск финальной доочистки списка признаков от штатов и сезонов...

=== ИТОГОВЫЙ ЭТАЛОННЫЙ КУЛИНАРНЫЙ СПИСОК ===
было... колок после точечной nlp валидации: 366
осталось ингредиентов-столбцов для модели: 347


In [23]:
# находим штаты которые реально присутствовали в valid_ingredient_columns и теперь удаляются
removed_states = [
    col for col in valid_ingredient_columns if col.lower().strip() in us_states
]

print("СПИСОК УДАЛЕННЫХ ГЕОГРАФИЧЕСКИХ ШТАТОВ")
print(f"всего удалено штатов: {len(removed_states)}")
print(removed_states)


СПИСОК УДАЛЕННЫХ ГЕОГРАФИЧЕСКИХ ШТАТОВ
всего удалено штатов: 7
['california', 'florida', 'idaho', 'new mexico', 'new york', 'vermont', 'wisconsin']


In [28]:
must_be_missing = [
    "title",
    "leftovers",
    "california",
    "dominican republic",
    "protein",
    "low cholesterol",
    "pan-fry",
    "fruit",
    "stew",
    "braise",
]

must_be_present = ["milk", "jam"]

for col in must_be_missing:
    if col not in perfect_ingredient_columns:
        print(f"{col} верно")
    else:
        print(f"{col} облом")

title верно
leftovers верно
california верно
dominican republic верно
protein верно
low cholesterol верно
pan-fry верно
fruit верно
stew верно
braise верно


In [29]:
print(f"осталось ингредиентов-столбцов для модели: {len(perfect_ingredient_columns)}")

осталось ингредиентов-столбцов для модели: 347


In [30]:
from sklearn.feature_selection import VarianceThreshold

print("запуск анализа редких признаков через дисперсию...")

# создание чистой числовой матрицы х на основе 349 колонок
# perfect_ingredient_columns — это список имен колонок, который мы получили ранее
X_matrix = df_orig[perfect_ingredient_columns].astype(np.float32)

#  минимальный порог встречаемости ингредиента
# напр., вполне можно убрать продукты, которые встречаются реже, чем в 10 рецептах из 20 000
min_recipes = 10
total_recipes = len(X_matrix)
p_threshold = min_recipes / total_recipes
variance_threshold_value = p_threshold * (1 - p_threshold)

# инициализация и обучение селектора дисперсии
selector = VarianceThreshold(threshold=variance_threshold_value)
selector.fit(X_matrix)

# списки с колнками проходящими и непроходящими фильтр
supported_mask = selector.get_support()
columns_to_keep = [
    col for col, keep in zip(perfect_ingredient_columns, supported_mask) if keep
]
columns_to_drop = [
    col for col, keep in zip(perfect_ingredient_columns, supported_mask) if not keep
]

# отчет по выбрасываемым ингредиентам для контроля
drop_report = []
for col in columns_to_drop:
    occurrence_count = int(X_matrix[col].sum())
    drop_report.append(
        {"Ингредиент": col, "Встречаемость (кол-во рецептов)": occurrence_count}
    )

df_drop_report = pd.DataFrame(drop_report).sort_values(
    by="Встречаемость (кол-во рецептов)", ascending=False
)

print("\n=== ИТОГИ АНАЛИЗА ДИСПЕРСИИ ===")
print(f"исходное число чистых кулинарных колонок: {len(perfect_ingredient_columns)}")
print(f"будет удалено слишком редких признаков: {len(columns_to_drop)}")
print(f"останется плотных признаков для модели: {len(columns_to_keep)}")
print("-" * 50)

if not df_drop_report.empty:
    print("точный список ингредиентов, которые предлагает выкинуть алгоритм:")
    print(df_drop_report.to_string(index=False))
else:
    print("при таком пороге ни один ингредиент не будет удален.")


запуск анализа редких признаков через дисперсию...

=== ИТОГИ АНАЛИЗА ДИСПЕРСИИ ===
исходное число чистых кулинарных колонок: 347
будет удалено слишком редких признаков: 43
останется плотных признаков для модели: 304
--------------------------------------------------
точный список ингредиентов, которые предлагает выкинуть алгоритм:
         Ингредиент  Встречаемость (кол-во рецептов)
              gouda                               10
              pizza                               10
          mandoline                               10
        marshmallow                                9
             lychee                                8
               taco                                8
               tart                                8
            buffalo                                8
            octopus                                7
            poblano                                7
            biscuit                                7
  pomegranate juice           

In [31]:
print("финишная ручная фильтрация аномалий")

# вносим список вручную чтобы не срезать "gouda", "rabbit", "octopus", "poblano", "lychee", "sourdough" итп
proven_garbage_variance = {
    "mandoline",
    "skewer",
    "boston",
    "mexico",
    "denver",
    "pizza",
    "taco",
    "tart",
    "muffin",
    "lasagna",
    "potato salad",
    "waffle",
    "macaroni and cheese",
}

perfect_ingredient_columns = [
    col
    for col in perfect_ingredient_columns
    if col.lower().strip() not in proven_garbage_variance
]

print(f"осталось ингредиентов для модели: {len(perfect_ingredient_columns)}")

финишная ручная фильтрация аномалий
осталось ингредиентов для модели: 334


In [36]:
print("сохранение финального перечня колонок в файл...")
with open("../datasets/perfect_ingredient_columns.json", "w", encoding="utf-8") as f:
    json.dump(perfect_ingredient_columns, f, ensure_ascii=False, indent=4)

print("файл perfect_ingredient_columns.json успешно записан в папку datasets")
print(f"в него сохранено ровно: {len(perfect_ingredient_columns)} ингредиента")

сохранение финального перечня колонок в файл...
файл perfect_ingredient_columns.json успешно записан в папку datasets
в него сохранено ровно: 334 ингредиента


In [33]:
print("Очистка дубликатов по названиям рецептов и рецептов < 2 ингредиентов")
print("-" * 75)

df_sync = df_orig.copy()
df_sync["title"] = df_sync["title"].astype(str).str.lower().str.strip()
df_clean_titles = df_sync.drop_duplicates(subset=["title"])
ingredient_counts = df_clean_titles[perfect_ingredient_columns].sum(axis=1)
df_final_rows = df_clean_titles[ingredient_counts > 1].copy()

print(f"Исходный объем сырого датасета: {df_orig.shape}")
print(
    f"кол-во строк датасета после удаления дубликатов по тайтлам и рецептов с < 2 ингредиентами: {df_final_rows.shape[0]}"
)

Очистка дубликатов по названиям рецептов и рецептов < 2 ингредиентов
---------------------------------------------------------------------------
Исходный объем сырого датасета: (20052, 680)
кол-во строк датасета после удаления дубликатов по тайтлам и рецептов с < 2 ингредиентами: 17261


In [34]:
print("Анализ и удаление экстремальных выбросов по калорийности...")

huge_cal_mask = df_final_rows["calories"] > 10000

top_huge_cal = (
    df_final_rows[huge_cal_mask][["title", "calories"]]
    .sort_values(by="calories", ascending=False)
    .head(10)
    .values.tolist()
)

# Очищаем строки: удаляем гигантов, но сохраняем все NaN
df_step2_calories = df_final_rows[~huge_cal_mask | df_final_rows["calories"].isna()]

print(f"Исходный объем на этом шаге: {df_final_rows.shape[0]} строк")
print(
    f"Удалено строк с выбросами калорийости: {len(df_final_rows) - len(df_step2_calories)}"
)

if top_huge_cal:
    print("\nПримеры выбросов:")
    for title, cal in top_huge_cal:
        print(f"  - '{title}' ({int(cal)} ккал)")

print(
    f"\nРазмер датасета после удаления калорийных выбросов: {df_step2_calories.shape[0]} строк"
)


Анализ и удаление экстремальных выбросов по калорийности...
Исходный объем на этом шаге: 17261 строк
Удалено строк с выбросами калорийости: 16

Примеры выбросов:
  - 'pear-cranberry mincemeat lattice pie' (30111218 ккал)
  - 'deep-dish wild blueberry pie' (29997918 ккал)
  - 'apricot, cranberry and walnut pie' (13062948 ккал)
  - 'lamb köfte with tarator sauce' (4518216 ккал)
  - 'rice pilaf with lamb, carrots, and raisins' (4157357 ккал)
  - 'chocolate-almond pie' (3358273 ккал)
  - 'caramelized apple and pear pie' (3358029 ккал)
  - 'merguez lamb patties with golden raisin couscous' (54512 ккал)
  - 'grilled lamb chops with porcini mustard' (24117 ккал)
  - 'grilled cumin chicken breasts with avocado salsa' (22312 ккал)

Размер датасета после удаления калорийных выбросов: 17245 строк


In [35]:
print("Финальное сжатие матрицы до 334 колонок (Сохраняем все рецепты)...")
print("-" * 75)

# Просто урезаем наш массив строк df_step2_calories строго до 334 выверенных колонок-ингредиентов
df_final_matrix = df_step2_calories[perfect_ingredient_columns]

print(f"Исходный объем на этом шаге: {df_step2_calories.shape}")
print(f"итоговый дф: {df_final_matrix.shape}")

Финальное сжатие матрицы до 334 колонок (Сохраняем все рецепты)...
---------------------------------------------------------------------------
Исходный объем на этом шаге: (17245, 680)
итоговый дф: (17245, 334)
